In [1]:
import numpy as np
import quantile_forest as qrf
from itertools import product
# https://stackoverflow.com/questions/42212810/tqdm-in-jupyter-notebook-prints-new-progress-bars-repeatedly:
from tqdm.notebook import tqdm

# Pirmoji dalis: COG išvesties modelis

In [2]:
X_train = np.load('../X_train_COGoutput.npy')
y_train = np.load('../y_train_COGoutput.npy')

X_val = np.load('../X_val_COGoutput.npy')
y_val = np.load('../y_val_COGoutput.npy')

> Išimame COG autoregresyvias reikšmes

In [3]:
X_train = np.concatenate( [X_train[:, :, [0]], X_train[:, :, 2:]], axis=2)
X_val = np.concatenate( [X_val[:, :, [0]], X_val[:, :, 2:]], axis=2)

#### Performuojame, kad galima būtų naudoti kaip ML modelio įvestį

In [4]:

reshape_params = (-1, X_train.shape[1]*X_train.shape[2]) 

X_train = X_train.reshape(reshape_params)
X_val = X_val.reshape(reshape_params)

y_train = y_train[:, 0].ravel()
y_val = y_val[:, 0].ravel()

In [19]:
params = {
    'n_estimators': [8, 16, 32],
    'max_depth': [None, 16, 64],
}

param_sets = list( product(*params.values()) )

In [20]:
def evaluate_metrics(
    qrf_model: qrf.RandomForestQuantileRegressor,
    X_val: np.ndarray, y_val: np.ndarray
) -> tuple:

    model_output = qrf_model.predict(X_val, quantiles=[0.025, 0.975])

    y_pred_low = model_output[:, 0]
    y_pred_high = model_output[:, 1]

    picp = np.mean( 
        (y_pred_low <= y_val) &
        (y_val <= y_pred_high)
    )

    pinaw = .5 * np.mean(
        np.abs(y_pred_high - y_pred_low)
    )

    return {
        'picp': picp,
        'pinaw': pinaw
    }

In [ ]:
qrf_model = qrf.RandomForestQuantileRegressor(
    n_estimators=8,
    max_depth=None,
    n_jobs=-1,
    verbose=True
)

qrf_model.fit(X_train, y_train)

metrics = evaluate_metrics(qrf_model, X_val, y_val)

[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 12 concurrent workers.


In [25]:
grid_search_results = {
    'param_set': [],
    'picp': [],
    'pinaw': []
}

for param_set in param_sets:
    n_est, max_d = param_set

    qrf_model = qrf.RandomForestQuantileRegressor(
        n_estimators=n_est,
        max_depth=max_d,
        n_jobs=-1,
        verbose=True
    )

    qrf_model.fit(X_train, y_train)

    metrics = evaluate_metrics(qrf_model, X_val, y_val)

    grid_search_results['param_set'].append(param_set)    
    grid_search_results['picp'].append(
        metrics['picp']
    )    
    grid_search_results['pinaw'].append(
        metrics['pinaw']
    )    

    print( '='*10 )
    print( param_set )
    print( metrics )
    print( '='*10 )

[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 12 concurrent workers.


KeyboardInterrupt: 

In [ ]:
grid_search_results

In [ ]:
qrf_model.predict(X_val, quantiles=[0.025, 0.971])